In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import psycopg2
from math import radians, cos, sin, asin, sqrt
import plotly.graph_objects as go
import plotly.express as p

pd.set_option('display.max_columns', 100)
load_dotenv()

True

In [3]:
df = pd.read_csv('GeoObjectsV2_3Mar2020.csv', encoding = "ISO-8859-1")

In [4]:
df.head()

,Obj_ID,Obj_ParentID,Obj_CountryID,Obj_TypeID,Obj_Code,Obj_Code2,Obj_StateCode,Obj_AirportCodes,Obj_Name,Obj_Keywords,Obj_GIATA,Obj_FBLocationID,Obj_Priority,Obj_StopParent,Obj_IsDeleted,Obj_UpdatedDateTime,Obj_CreatedDateTime,Obj_SourceRef,Obj_Description,Obj_WeatherCode,Obj_CurrencyCode,Obj_Lon,Obj_Lat,Obj_Geo,Obj_MapZoom,Obj_Alternatives,Obj_Path,Obj_IsHidden,Obj_NamePath,Obj_AllowOnCombinedDP,Obj_AllowOnNonPH,Obj_GlobalHidden,Obj_Active
0,1,0,0,1,NaN,NaN,NaN,NaN,Africa,NaN,NaN,NaN,0,0,0,2009-03-09 22:58,2009-03-09 22:58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/1/,0,/Africa,1,1,0,1
1,2,0,0,1,NaN,NaN,NaN,NaN,Asia,NaN,NaN,NaN,0,0,0,2009-03-09 22:58,2009-03-09 22:58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/2/,0,/Asia,1,1,0,1
2,3,0,0,1,NaN,NaN,NaN,NaN,Australasia,NaN,NaN,NaN,0,0,0,2009-03-09 22:58,2009-03-09 22:58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/3/,0,/Australasia,1,1,0,1
3,4,0,0,1,NaN,NaN,NaN,NaN,Europe,NaN,NaN,NaN,0,0,0,2009-03-09 22:58,2009-03-09 22:58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/4/,0,/Europe,1,1,0,1
4,5,0,0,1,NaN,NaN,NaN,NaN,Middle East,NaN,NaN,NaN,0,0,0,2009-03-09 22:58,2009-03-09 22:58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/5/,0,/Middle East,1,1,0,1


In [5]:
df = df.dropna(axis=0, subset=['Obj_AirportCodes', 'Obj_Name', 'Obj_Lon', 'Obj_Lat'], how='any')

In [6]:
df.sample(5)

,Obj_ID,Obj_ParentID,Obj_CountryID,Obj_TypeID,Obj_Code,Obj_Code2,Obj_StateCode,Obj_AirportCodes,Obj_Name,Obj_Keywords,Obj_GIATA,Obj_FBLocationID,Obj_Priority,Obj_StopParent,Obj_IsDeleted,Obj_UpdatedDateTime,Obj_CreatedDateTime,Obj_SourceRef,Obj_Description,Obj_WeatherCode,Obj_CurrencyCode,Obj_Lon,Obj_Lat,Obj_Geo,Obj_MapZoom,Obj_Alternatives,Obj_Path,Obj_IsHidden,Obj_NamePath,Obj_AllowOnCombinedDP,Obj_AllowOnNonPH,Obj_GlobalHidden,Obj_Active
133939,142064,1473,17,6,JUJ,NaN,NaN,JUJ,Libertador General San Martín,NaN,NaN,NaN,0,0,0,2019-04-30 14:48,2019-04-30 14:48,NaN,NaN,NaN,NaN,-64.792826,-23.815332,0xE6100000010CC66D3480B7D037C03411363CBD3250C0,NaN,NaN,NaN,0,NaN,1,1,0,1
105450,113512,177,25,10,BGI,NaN,NaN,BGI,Harry Bayley Observatory,NaN,NaN,NaN,0,0,0,2016-02-29 18:42,2016-02-29 15:32,6061264,NaN,NaN,NaN,-59.584286,13.088131,0xE6100000010C228E75711B2D2A40C286A757CACA4DC0,NaN,NaN,/113512/177/17489/18571/25/7/,0,NaN,1,1,0,1
5951,8256,89,89,5,MSU,NaN,NaN,MSU,Maseru,NaN,NaN,NaN,0,0,0,2009-03-27 16:12,2009-03-27 16:12,NaN,NaN,NaN,NaN,27.483333,-29.316667,0xE6100000010CA9A44E4013513DC024287E8CB97B3B40,NaN,NaN,/8256/89/1/,0,/Maseru/Lesotho/Africa,1,1,0,1
50098,56245,1047,20,5,ADL,NaN,NaN,ADL,Ovingham,NaN,NaN,NaN,0,1,0,2013-06-18 18:06,2013-06-18 18:06,NaN,NaN,NaN,NaN,138.584349,-34.895911,0xE6100000010C423EE8D9AC7241C0A69BC420B0526140,NaN,NaN,/56245/1047/20/3/,0,NaN,1,1,0,1
77093,85155,1093,158,10,LAX,NaN,NaN,LAX,El Capitan Theatre,NaN,NaN,NaN,0,0,0,2016-02-29 18:13,2016-02-29 14:55,6159836,NaN,NaN,NaN,-118.339859,34.101549,0xE6100000010CA245B6F3FD0C4140F6285C8FC2955DC0,NaN,NaN,/85155/1093/17972/18232/1000/158/6/,0,NaN,1,1,0,1


In [7]:
df['Obj_Name'].value_counts()

Plaza de Armas                21
Central Park                  19
Olympic Stadium               16
Museum of Contemporary Art    16
National Museum               16
                              ..
Hostos Park                    1
Saint Mary                     1
Luabo                          1
Barcelona Egyptian Museum      1
Nash Farm                      1
Name: Obj_Name, Length: 84479, dtype: int64

There are many common names.

In [8]:
df.loc[df['Obj_Name'] == 'Central Park'][['Obj_AirportCodes', 'Obj_Name', 'Obj_Lon', 'Obj_Lat']]

,Obj_AirportCodes,Obj_Name,Obj_Lon,Obj_Lat
11123,JFK,Central Park,-73.965355,40.782865
52009,"MEL,HLT",Central Park,133.775136,-25.274398
70194,NYC,Central Park,-73.965028,40.783700
70324,NBO,Central Park,36.817137,-1.286460
71278,YVR,Central Park,-123.020956,49.222019
91984,KHH,Central Park,120.298098,22.623900
101706,CJS,Central Park,-106.426803,31.684987
105012,"HOU,IAH",Central Park,-100.963334,35.540597
108639,CGK,Central Park,106.790907,-6.176385
109291,AHN,Central Park,-95.855262,32.194472


In [9]:
df['Obj_Active'].value_counts()

1    90045
0       18
Name: Obj_Active, dtype: int64

Don't know why there are inactive Obj, but not many, self-decided to include them in the analysis.

Use airport data and seat capacity data, then keep the airports we are interest.

In [10]:
query = """
    SELECT "code", "lat", "lng", "name", "country_name"
    FROM airport_locations
    where lat is not null
"""
db_url = f'postgresql://{os.environ["PG_USER"]}:{os.environ["PG_PASSWORD"]}@{os.environ["PG_HOST"]}/dataportal'
airports_df = pd.read_sql_query(query, db_url)

In [11]:
# EC's code
query = """
    SELECT "Dest", "Airline Type", SUM("Seats") as "Annual Seats"
    FROM capacity
    GROUP BY "Dest", "Airline Type"
    ORDER BY "Dest"
"""
db_url = f'postgresql://{os.environ["PG_USER"]}:{os.environ["PG_PASSWORD"]}@{os.environ["PG_HOST"]}/dataportal'
seats_df = pd.read_sql_query(query, db_url)
# only include the airline types we care about
# aggregate by Dest IATA
seats_df = (
    seats_df
    .loc[seats_df['Airline Type'].isin(['Scheduled Passenger', 'Scheduled Passenger/Cargo'])]
    .groupby(['Dest'])
    .sum()
)

# Only keep the airports we care about
airports_df = airports_df[airports_df.code.isin(seats_df.index)]

In [12]:
airports_df.head()

,code,lat,lng,name,country_name
0,AAA,-17.355101,-145.508499,Anaa,French Polynesia
4,AAE,36.822201,7.809400,Rabah Bitat,Algeria
11,AAL,57.093102,9.850000,Aalborg,Denmark
13,AAN,24.261700,55.609200,Al Ain International,United Arab Emirates
15,AAP,-0.371900,117.257202,Aji Pangeran Tumenggung Airport,Indonesia


In [13]:
df = df[['Obj_AirportCodes', 'Obj_Name', 'Obj_Lon', 'Obj_Lat']]

In [14]:
# Separate rows
# https://stackoverflow.com/questions/12680754/split-explode-pandas-dataframe-string-entry-to-separate-rows

def explode(df, lst_cols, fill_value='', preserve_index=False):
    # make sure `lst_cols` is list-alike
    if (lst_cols is not None
        and len(lst_cols) > 0
        and not isinstance(lst_cols, (list, tuple, np.ndarray, pd.Series))):
        lst_cols = [lst_cols]
    # all columns except `lst_cols`
    idx_cols = df.columns.difference(lst_cols)
    # calculate lengths of lists
    lens = df[lst_cols[0]].str.len()
    # preserve original index values    
    idx = np.repeat(df.index.values, lens)
    # create "exploded" DF
    res = (pd.DataFrame({
                col:np.repeat(df[col].values, lens)
                for col in idx_cols},
                index=idx)
             .assign(**{col:np.concatenate(df.loc[lens>0, col].values)
                            for col in lst_cols}))
    # append those rows that have empty lists
    if (lens == 0).any():
        # at least one list in cells is empty
        res = (res.append(df.loc[lens==0, idx_cols], sort=False)
                  .fillna(fill_value))
    # revert the original index order
    res = res.sort_index()
    # reset index if requested
    if not preserve_index:        
        res = res.reset_index(drop=True)
    return res

In [15]:
df = explode(df.assign(Obj_AirportCodes=df.Obj_AirportCodes.str.split(',')), 'Obj_AirportCodes')

In [16]:
df.tail()

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes
133469,48.469811,7.726144,Plobsheim,SXB
133470,48.658010,7.827133,La Wantzenau,BSL
133471,48.658010,7.827133,La Wantzenau,SXB
133472,48.829742,7.963965,Soufflenheim,BSL
133473,48.829742,7.963965,Soufflenheim,SXB


In [17]:
# merge POI df with airports df

result = pd.merge(df, airports_df, how='inner', left_on='Obj_AirportCodes', right_on='code').drop('code', axis=1)

In [18]:
result.sample(5)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name
11907,45.430255,12.325808,Squero di San Trovaso,TSF,45.683300,12.083300,Venice-Treviso,Italy
41034,53.798900,-1.542070,City Varieties Music Hall,LPL,53.338299,-2.866700,John Lennon Airport,United Kingdom
9813,28.665344,77.230282,St. James Church,AGR,27.158300,77.962799,Agra Airport,India
87095,51.454568,7.012340,Grillo-Theater,DTM,51.516701,7.466700,Dortmund Airport,Germany
46821,45.462840,-122.656984,Ping Pong's Pint Sized Puppet Museum,PWM,43.644402,-70.303596,Portland Intl Jetprt,United States


In [19]:
# get the distance between 2 GPS points
# Source: https://stackoverflow.com/questions/4913349/haversine-formula-in-python-bearing-and-distance-between-two-gps-points

def distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)

    All args must be of equal length.    

    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2

    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c  # 6371 is Radius of earth in kilometers. Use 3956 for miles
    return km

In [20]:
result['distance_km'] = distance(result['Obj_Lat'], result['Obj_Lon'], result['lat'], result['lng'])

In [21]:
result.loc[result['Obj_Name'] == "Santa Fe Children's Museum"]

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
49893,35.673369,-105.936517,Santa Fe Children's Museum,ABQ,35.041698,-106.606399,International Sunport,United States,77.042414
49926,35.673369,-105.936517,Santa Fe Children's Museum,YES,30.683300,51.545200,Yasouj Airport,Iran,17500.423426


In [22]:
result.loc[result['Obj_Name'] == "Stillwater"]

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
55507,42.93841,-73.653173,Stillwater,OKC,35.393101,-97.600601,Will Rogers World,United States,2657.768623
55666,42.93841,-73.653173,Stillwater,TUL,36.197800,-95.889702,Tulsa International,United States,2469.226142


In [23]:
result.loc[result['Obj_Name'] == "CN Tower"]

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
55723,56.130366,-106.346771,CN Tower,YYZ,43.679699,-79.625,Toronto Lester B. Pearson Intl,Canada,2954.391168
55746,43.642566,-79.387679,CN Tower,YYZ,43.679699,-79.625,Toronto Lester B. Pearson Intl,Canada,26.399602


In [24]:
result.loc[result['Obj_Name'] == "Grand Central Station"]

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
53663,34.16184,-118.283932,Grand Central Station,JFK,40.638599,-73.776901,John F Kennedy Intl,United States,4941.272435


In [25]:
result.loc[(result['country_name'] == 'United States') & (result['distance_km'] <= 300)].sort_values(by=['distance_km'], ascending=False).head(50)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
69947,37.034861,-78.897986,Red Hill Patrick Henry National Memorial,ORF,36.895802,-76.201401,Norfolk Intl,United States,299.864915
35578,37.715459,-119.677077,Tunnel View,SFO,37.619400,-122.372803,San Francisco Intl,United States,299.801485
64677,38.348371,-89.375818,Nashville Memorial Park,BNA,36.126701,-86.681900,Nashville International,United States,299.614250
54938,36.420747,-81.470609,Museum of Ashe County History,RDU,35.875301,-78.781898,Raleigh-Durham International,United States,299.148488
54939,36.434680,-81.468539,Ash County Park,RDU,35.875301,-78.781898,Raleigh-Durham International,United States,298.927685
107894,40.664733,-74.387481,Bowcraft Amusement Park (Bowcraft Playland),DCA,38.851700,-77.036903,Ronald Reagan National,United States,298.738845
35021,36.313435,-119.772944,Heritage Park,SFO,37.619400,-122.372803,San Francisco Intl,United States,298.633747
35272,34.600450,-120.193083,Mosby Winery,SFO,37.619400,-122.372803,San Francisco Intl,United States,298.481958
35430,37.213358,-119.699434,Picayune Rancheria of the Chukchansi Indians,SFO,37.619400,-122.372803,San Francisco Intl,United States,298.173548
43213,30.520634,-83.991277,Letchworth Mounds,MCO,28.431700,-81.324699,Orlando Intl,United States,297.943328


In [70]:
result.loc[result['country_name'] == 'Canada'].sort_values(by=['distance_km'], ascending=False).head(50)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
53632,-25.274398,133.775136,Sydney surrounds,YQY,46.166698,-60.049999,J.A. Douglas McCurdy,Canada,15273.211925
75635,-33.445061,149.554473,Mount Panorama Winery,ZBF,47.633301,-65.733299,Bathurst,Canada,13468.395086
75636,-33.439235,149.561857,National Motor Racing Museum,ZBF,47.633301,-65.733299,Bathurst,Canada,13467.937128
75632,-33.417653,149.581031,Bathurst,ZBF,47.633301,-65.733299,Bathurst,Canada,13467.041320
75634,-33.418171,149.586898,Bathurst Visitor Information Centre,ZBF,47.633301,-65.733299,Bathurst,Canada,13466.443426
33904,0.000000,0.000000,Save on Foods Memorial Centre,YVR,49.187199,-123.185303,Vancouver Intl,Canada,12338.361622
34002,0.000000,0.000000,Save on Foods Memorial Centre,CXH,49.299999,-123.133301,Harbour SPB,Canada,12329.414729
80689,0.000000,0.000000,Bell MTS Place (Manitoba),YWG,49.900002,-97.233299,J.A. Richardson Intl,Canada,10524.809049
56613,-76.016094,-30.234375,Tremblant,YUL,45.463902,-73.750000,Pierre E. Trudeau Intl,Canada,7680.607785
55427,60.516009,-150.701988,Bings Landing,YQG,42.274200,-82.963898,Windsor,Canada,7495.129243


The worst were all seem to be completely wrong airports.

In [26]:
result.loc[result['Obj_Name'] == 'Central Park']

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
10265,-6.176385,106.790907,Central Park,CGK,-6.125800,106.657799,Soekarno-Hatta Intl,Indonesia,14.889199
14330,-1.286460,36.817137,Central Park,NBO,-1.316700,36.930801,Jomo Kenyatta Intl,Kenya,12.921971
33230,51.174278,-115.572894,Central Park,YYC,51.112801,-114.019699,Calgary Intl,Canada,172.731165
33675,49.222019,-123.020956,Central Park,YVR,49.187199,-123.185303,Vancouver Intl,Canada,18.396455
35202,37.339995,-121.977527,Central Park,SFO,37.619400,-122.372803,San Francisco Intl,United States,46.963150
35775,37.339995,-121.977527,Central Park,OAK,37.723598,-122.215599,Metro Oakland Intl,United States,34.849115
53672,40.782865,-73.965355,Central Park,JFK,40.638599,-73.776901,John F Kennedy Intl,United States,21.423700
65323,35.540597,-100.963334,Central Park,IAH,29.983299,-95.339996,George Bush Intercontinental,United States,630.672302
66034,35.540597,-100.963334,Central Park,HOU,29.645000,-95.278603,William P Hobby,United States,638.037832
66563,36.619265,-88.310274,Central Park,SLC,40.784199,-111.966904,Salt Lake City Intl,United States,2630.034655


3 Central Parks share the same location but different airports and none of them makes sense. 

In [27]:
us = result.loc[result['country_name'] == 'United States']
fig = px.histogram(us, x="distance_km")
fig.show()

In [28]:
fig = go.Figure(data=[go.Histogram(x=us['distance_km'], histnorm='probability')])
fig.update_layout(title_text='POI and airport pairs distance probability distribution in US')
fig.show()

In [29]:
us.loc[us['distance_km'] == 0]

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
112804,33.450298,-88.5914,Golden Triangle Reg.,GTR,33.450298,-88.5914,Golden Triangle Regl,United States,0.0


In [30]:
us['distance_km'].describe()

count    27070.000000
mean       554.259701
std       1364.121451
min          0.000000
25%         12.622079
50%         41.092234
75%        240.135501
max      17856.248228
Name: distance_km, dtype: float64

In [31]:
us.loc[us['distance_km'] <= 250].sort_values(by=['distance_km'], ascending=False).head(50)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
45482,44.682004,-85.662344,Wuerfel Park,ORD,41.978100,-87.906097,O'Hare International,United States,249.994160
92165,13.444167,142.546309,Guam,GUM,13.481700,144.793304,Antonio B. Won Pat International Airport,United States,249.877157
45782,43.039285,-85.659907,Fifth Third Ballpark,ORD,41.978100,-87.906097,O'Hare International,United States,249.842054
49144,35.352481,-96.935721,Shawnee Little Theatre,MCI,39.292801,-94.726700,Kansas City Intl,United States,249.489481
109361,39.823490,-89.012810,Rock Springs Conservation Area,HSV,34.642799,-86.775597,International-Carl T. Jones Field,United States,249.412029
65949,30.954726,-97.515599,Mill Creek Golf Club,HOU,29.645000,-95.278603,William P Hobby,United States,249.255242
34529,40.365839,-120.651074,Diamond Mountain Golf Club,SFO,37.619400,-122.372803,San Francisco Intl,United States,249.218710
66086,26.072322,-97.485419,Little Graceland,HOU,29.645000,-95.278603,William P Hobby,United States,249.210748
65478,30.505310,-97.579180,Star Ranch Golf Club,IAH,29.983299,-95.339996,George Bush Intercontinental,United States,249.068921
106872,39.211160,-106.946080,Snowmass Ice Age Discovery Center,COS,38.808899,-104.709198,Municipal,United States,249.027479


In [32]:
ca = result.loc[result['country_name'] == 'Canada']
fig = go.Figure(data=[go.Histogram(x=ca['distance_km'], histnorm='probability')])
fig.update_layout(title_text='POI and airport pairs distance probability distribution in Canada')
fig.show()

In [33]:
ca['distance_km'].describe()

count     5211.000000
mean       229.892908
std        803.737055
min          0.000000
25%         13.479291
50%         48.297635
75%        139.779425
max      15273.211925
Name: distance_km, dtype: float64

In [34]:
ca.sort_values(by=['distance_km'], ascending=False).head(50)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
53632,-25.274398,133.775136,Sydney surrounds,YQY,46.166698,-60.049999,J.A. Douglas McCurdy,Canada,15273.211925
75635,-33.445061,149.554473,Mount Panorama Winery,ZBF,47.633301,-65.733299,Bathurst,Canada,13468.395086
75636,-33.439235,149.561857,National Motor Racing Museum,ZBF,47.633301,-65.733299,Bathurst,Canada,13467.937128
75632,-33.417653,149.581031,Bathurst,ZBF,47.633301,-65.733299,Bathurst,Canada,13467.041320
75634,-33.418171,149.586898,Bathurst Visitor Information Centre,ZBF,47.633301,-65.733299,Bathurst,Canada,13466.443426
33904,0.000000,0.000000,Save on Foods Memorial Centre,YVR,49.187199,-123.185303,Vancouver Intl,Canada,12338.361622
34002,0.000000,0.000000,Save on Foods Memorial Centre,CXH,49.299999,-123.133301,Harbour SPB,Canada,12329.414729
80689,0.000000,0.000000,Bell MTS Place (Manitoba),YWG,49.900002,-97.233299,J.A. Richardson Intl,Canada,10524.809049
56613,-76.016094,-30.234375,Tremblant,YUL,45.463902,-73.750000,Pierre E. Trudeau Intl,Canada,7680.607785
55427,60.516009,-150.701988,Bings Landing,YQG,42.274200,-82.963898,Windsor,Canada,7495.129243


In [35]:
ca.loc[ca['distance_km'] <= 250].sort_values(by=['distance_km'], ascending=False).head(50)

,Obj_Lat,Obj_Lon,Obj_Name,Obj_AirportCodes,lat,lng,name,country_name,distance_km
33319,50.446793,-116.249903,Greywolf Golf Course,YYC,51.112801,-114.019699,Calgary Intl,Canada,249.970685
57837,45.710360,-73.622144,Multiglace,YQB,46.789200,-71.398102,Jean Lesage Intl,Canada,249.906112
80417,50.327661,-64.037784,Mingan,YZV,50.216702,-66.283302,Sept-Iles,Canada,249.743972
33316,50.447290,-116.246240,Sunbird Chair Lift,YYC,51.112801,-114.019699,Calgary Intl,Canada,249.563405
33315,50.456580,-116.241590,Toby Chair Lift,YYC,51.112801,-114.019699,Calgary Intl,Canada,248.995262
79624,50.327661,-64.037784,Mingan,YNA,50.183300,-61.799999,Natashquan,Canada,248.937385
33313,50.458700,-116.239920,Village Gondola,YYC,51.112801,-114.019699,Calgary Intl,Canada,248.798469
33134,50.458341,-116.238962,Panorama,YYC,51.112801,-114.019699,Calgary Intl,Canada,248.694862
57899,45.481854,-73.599725,Westmount Park and Library,YQB,46.789200,-71.398102,Jean Lesage Intl,Canada,248.667079
33218,50.460552,-116.238275,Panorama Mountain Ski Area,YYC,51.112801,-114.019699,Calgary Intl,Canada,248.606025
